<a href="https://colab.research.google.com/github/Theno1gitty/FIFA-World-Cup-Goal-Records-A-Data-Driven-Explanation-on-Why-Broken-at-such-a-Fast-Rate/blob/main/WC_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### FIFA World Cup Goal Records: A Data-Driven Explanation on Why Goal-Scoring Records are Being Broken at such a Fast Rate

I made this Python project in the summer of 2026, during the FIFA World Cup.

As I watched Messi score his first World Cup hattrick, I noticed that the previous goal-scoring record set by Miroslav Klose — a player who played during the 2000s — towards the end of his career had already been broken by Messi on the way to his treble, and there was still a huge chunk of games to be played! To make things more astonishing, several other players who were currently playing in the World Cup were already on the leaderboard too!

This led me to wonder, why are football-related records being broken so easily, when the skill-level of players relative to each other has remained the same?

I realised that this could be answered with the help of data, and hence began writing this program

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
top_goalscorers = pd.read_csv("/content/drive/MyDrive/World Cup Data Analysis/wc_top_scorers.csv")
print(top_goalscorers)

In [ ]:
top_scorers = top_goalscorers.groupby('player')['goals'].sum().reset_index()
top_scorers.rename(columns={'goals': 'total_goals'}, inplace=True)
top_scorers = top_scorers.sort_values(by='total_goals', ascending=False)
display(top_scorers.style.hide(axis='index'))

This dataset was not what I wanted: I needed a dataset that recorded all the goals scored across each edition of the competition. I finally found the dataset I required in Kaggle, which I am viewing with the API provided:

In [ ]:
import kagglehub

path = kagglehub.dataset_download("jahaidulislam/fifa-world-cup-all-goals-1930-2022-dataset")

print("Path to dataset files:", path)

I encountered some problems when trying to read the file, so I asked Gemini for help

In [ ]:
import os

print(f"The provided path '{path}' is a directory.")
print("Contents of the directory:")
directory_contents = os.listdir(path)
for item in directory_contents:
    print(f"- {item}")

csv_filename = 'FIFA World Cup All Goals 1930-2022.csv'
full_csv_path = os.path.join(path, csv_filename)

if os.path.exists(full_csv_path):
    print(f"\nAttempting to read the CSV file: {full_csv_path}")
    try:
        historical_goalscoring = pd.read_csv(full_csv_path, encoding='latin1')
        print("Successfully loaded 'historical_goalscoring' DataFrame with 'latin1' encoding.")
        display(historical_goalscoring)
    except UnicodeDecodeError:
        print("UnicodeDecodeError: 'latin1' encoding also failed. Trying 'ISO-8859-1'.")
        historical_goalscoring = pd.read_csv(full_csv_path, encoding='ISO-8859-1')
        print("Successfully loaded 'historical_goalscoring' DataFrame with 'ISO-8859-1' encoding.")
        display(historical_goalscoring)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print(f"\nError: The expected CSV file '{csv_filename}' was not found at '{full_csv_path}'.")
    print("Please check the directory contents listed above and update 'csv_filename' variable with the correct file name.")

I then created a new dataframe where I displayed a "leaderboard" for top scorers in the FIFA World Cup. With some help from Gemini to remove the "not applicable" labels next to some players' names, I created the dataframe displayed below



In [ ]:
historical_goalscoring['player_name'] = historical_goalscoring['given_name'].fillna('') + ' ' + \
                                        historical_goalscoring['family_name'].fillna('')


historical_goalscoring['player_name'] = historical_goalscoring['player_name'].str.strip()

player_total_goals = historical_goalscoring.groupby('player_name').size().reset_index(name='total_goals')

player_total_goals = player_total_goals[~player_total_goals['player_name'].str.contains('not applicable', case=False)]

player_total_goals = player_total_goals.sort_values(by='total_goals', ascending=False)
player_matches = historical_goalscoring[['player_name', 'match_id']].drop_duplicates()
player_matches_count = player_matches.groupby('player_name')['match_id'].count().reset_index(name='total_matches')


player_total_goals = pd.merge(player_total_goals, player_matches_count, on='player_name', how='left')

display(player_total_goals.style.hide(axis='index'))

In [ ]:
player_total_goals['goals_per_game'] = player_total_goals['total_goals'] / player_total_goals['total_matches']
display(player_total_goals.sort_values(by='goals_per_game', ascending=False))

This is not a reliable attribute, as none of the players ranked in this way in the modified dataframe corresponds to the tournament's top scorers.

### Analyzing Trends in Total Goals and Matches Per World Cup Edition

Let's investigate the total number of goals scored and matches played in each FIFA World Cup edition. An increase in either of these could lead to more opportunities for players to set new goal-scoring records.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract year from 'tournament_name' and convert to integer
historical_goalscoring['year'] = historical_goalscoring['tournament_name'].str.extract(r'(\d{4})')[0].astype(int)

# Group by year to get total goals and matches per tournament
tournament_stats = historical_goalscoring.groupby('year').agg(
    total_goals=('goal_id', 'count'),
    total_matches=('match_id', 'nunique')
).reset_index()

# Calculate goals per match
tournament_stats['goals_per_match'] = tournament_stats['total_goals'] / tournament_stats['total_matches']

print("Tournament Statistics (Goals and Matches over time):")
display(tournament_stats)

Now, let's visualize these trends using line plots. We'll plot the total number of matches and the total number of goals over the years to see if there's an increasing trend.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot Total Matches Per Year
sns.lineplot(ax=axes[0], x='year', y='total_matches', data=tournament_stats, marker='o', color='skyblue')
axes[0].set_title('Total Matches Played Per FIFA World Cup Edition (1930-2022)')
axes[0].set_ylabel('Total Matches')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot Total Goals Per Year
sns.lineplot(ax=axes[1], x='year', y='total_goals', data=tournament_stats, marker='o', color='salmon')
axes[1].set_title('Total Goals Scored Per FIFA World Cup Edition (1930-2022)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Goals')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Plot Total Matches Per Year
sns.lineplot(ax=axes[0], x='year', y='total_matches', data=tournament_stats, marker='o', color='skyblue')
axes[0].set_title('Total Matches Played Per FIFA World Cup Edition (1930-2022)')
axes[0].set_ylabel('Total Matches')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot Total Goals Per Year
sns.lineplot(ax=axes[1], x='year', y='total_goals', data=tournament_stats, marker='o', color='salmon')
axes[1].set_title('Total Goals Scored Per FIFA World Cup Edition (1930-2022)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Total Goals')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


From these plots, we can observe that both the number of matches played and the total goals scored in each tournament have generally increased over time. More matches mean more opportunities for players to score, and a higher total number of goals in a tournament naturally leads to more records being broken.

Let's also look at the average goals per match, to see if the *rate* of scoring has changed, not just the absolute numbers.

In [ ]:
fig = plt.figure(figsize=(14, 6))
sns.lineplot(x='year', y='goals_per_match', data=tournament_stats, marker='o', color='lightgreen')
plt.title('Average Goals Per Match Per FIFA World Cup Edition (1930-2022)')
plt.xlabel('Year')
plt.ylabel('Average Goals Per Match')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(14, 6))
sns.lineplot(x='year', y='goals_per_match', data=tournament_stats, marker='o', color='lightgreen')
plt.title('Average Goals Per Match Per FIFA World Cup Edition (1930-2022)')
plt.xlabel('Year')
plt.ylabel('Average Goals Per Match')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


The 'Average Goals Per Match' plot shows some fluctuation but doesn't indicate a consistent, dramatic increase in scoring efficiency per game. This suggests that the rise in *total* goals and the breaking of records are more strongly tied to the *increased number of matches played* in modern tournaments rather than a significant increase in the average number of goals scored in each individual game. More games provide more chances for top players to accumulate goals over the course of a tournament and throughout their World Cup careers.

The graph above also provides an interesting statistic — playstyles have become more defensive.

Even then, modern-era top players have performed consistently well, allowing them to break records with relative ease

### Impact of Increasing Number of Participating Teams

One clear factor influencing the total number of matches and goals is the number of teams participating in the World Cup. More teams mean more group stage matches and potentially more knockout stage matches, leading to a higher overall game count per tournament.

In [ ]:
# Calculate the number of unique teams per year
teams_per_year = historical_goalscoring.groupby('year')['team_name'].nunique().reset_index(name='num_teams')

print("Number of Participating Teams Over Time:")
display(teams_per_year)

fig = plt.figure(figsize=(14, 6))
sns.lineplot(x='year', y='num_teams', data=teams_per_year, marker='o', color='purple')
plt.title('Number of Participating Teams Per FIFA World Cup Edition (1930-2022)')
plt.xlabel('Year')
plt.ylabel('Number of Teams')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

The plot above clearly shows a significant increase in the number of participating teams, especially from 16 to 24, and then to 32. This directly correlates with an increased number of matches, providing more opportunities for goal-scoring.

### Contribution of Top Players to Total Goals

Let's also consider if the top individual goal-scorers are simply playing more matches (due to more teams/matches in general) or if their individual scoring rate is also a significant factor. We can look at the proportion of total tournament goals scored by the top players over time.

In [ ]:
# Calculate total goals per tournament
total_goals_per_tournament = tournament_stats.set_index('year')['total_goals']

player_goals_yearly = historical_goalscoring.groupby(['year', 'player_name']).size().reset_index(name='goals_in_year')

player_goals_yearly['proportion_of_total'] = player_goals_yearly.apply(
    lambda row: row['goals_in_year'] / total_goals_per_tournament.get(row['year'], 1),
    axis=1
)

# For example, Miroslav Klose, Gerd Müller, Lionel Messi, Just Fontaine
selected_players = ['Miroslav Klose', 'Gerd Müller', 'Lionel Messi', 'Just Fontaine']

fig = plt.figure(figsize=(14, 7))
for player in selected_players:
    player_data = player_goals_yearly[player_goals_yearly['player_name'] == player]
    if not player_data.empty:
        sns.lineplot(x='year', y='goals_in_year', data=player_data, marker='o', label=player)

plt.title('Goals Scored by Selected Top Scorers Per FIFA World Cup Edition')
plt.xlabel('Year')
plt.ylabel('Goals Scored in Tournament')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Player')
plt.tight_layout()
plt.show()

This graph illustrates how individual top scorers contribute to the overall goal count. While specific players have had standout tournaments, the general trend indicates that a higher total number of goals across tournaments (as seen in earlier plots) is less about an extraordinary increase in individual player efficiency per game in the modern era and more about the increased opportunities these players get due to more matches in expanded tournaments, allowing them to accumulate higher overall totals over their careers.

In essence, players are breaking records faster because:
1.  More Matches: Expanded tournaments mean more games are played, directly increasing the total goal tally and individual player chances. More games means more opportunities for top players to showcase their skills and score.

2.  Sustained Excellence: While the 'goals per game' rate might fluctuate, the most talented players maintain a high level of performance across these increased opportunities, leading to faster record-breaking.

Therefore, my initial observation about records being broken easily is largely attributable to the evolving structure of the World Cup itself, which allows players to accumulate more goals, rather than a radical shift in player skill levels relative to each other.